In [1]:
from itertools import combinations

from tqdm import tqdm
import polars as pl
import torch
import numpy as np
import ot
import matplotlib.pyplot as plt
from jaxtyping import Float
from sklearn.neighbors import KernelDensity

from muutils.dbg import dbg_tensor

In [2]:
def load_pointclouds(
	path: str,
	pc_prefix: str = "pc.",
	n_dims: int | None = None,
) -> dict[str, Float[np.ndarray, "n_points dim"]]:
	"""Load point clouds from a JSONL file into NumPy arrays per model.

	# Parameters:
	 - `path: str`
	    Path to the JSONL file.
	 - `pc_prefix: str`
	    Prefix of principal-component columns to select (defaults to `"pc."`).
	 - `n_dims: Optional[int]`
	    Number of PC dimensions to load (starting from `pc.0`).
	    If `None`, all columns matching the prefix are loaded.

	# Returns:
	 - `dict[str, Float[np.ndarray, "n_points dim"]]`
	    A dict mapping each model name to an array of shape `(n_points, n_pcs)`.

	# Usage:

	```python
	>>> clouds = load_pointclouds("activations.jsonl", n_dims=10)
	>>> list(clouds.keys())
	['gemma-2b', 'other-model', ...]
	```
	"""
	df: pl.DataFrame = pl.read_ndjson(path)

	# Determine PC columns
	if n_dims is not None:
		pc_cols: list[str] = [f"{pc_prefix}{i}" for i in range(n_dims)]
	else:
		pc_cols = [c for c in df.columns if c.startswith(pc_prefix)]

	# Ensure columns exist
	missing = set(pc_cols) - set(df.columns)
	if missing:
		raise ValueError(f"Missing expected PC columns: {missing}")

	models: list[str] = df["activation.model"].unique().to_list()
	clouds: dict[str, np.ndarray] = {}
	for model in models:
		sub: pl.DataFrame = df.filter(pl.col("activation.model") == model)
		arr: np.ndarray = sub.select(pc_cols).to_numpy()
		clouds[model] = arr
	return clouds

In [3]:
def compute_wasserstein_matrix(
	clouds: dict[str, np.ndarray],
	sinkhorn_reg: float = 1e-1,
) -> Float[np.ndarray, "n_models n_models"]:
	"""Compute a symmetric matrix of squared 2-Wasserstein distances between point clouds,
	with a tqdm progress bar over model pairs.

	Uses entropic-regularized OT (Sinkhorn). Set `sinkhorn_reg=0` for exact EMD.

	# Parameters:
	 - `clouds: dict[str, np.ndarray]`
	    Mapping of model name → point-cloud array of shape (n_i, dim).
	 - `sinkhorn_reg: float`
	    Regularization strength; if 0 → exact EMD, if >0 → Sinkhorn.

	# Returns:
	 - `Float[np.ndarray, "n_models n_models"]`
	    A (MxM) matrix where entry (i,j) is W₂²(cloud_i, cloud_j).

	# Usage:

	```python
	>>> M = compute_wasserstein_matrix(clouds)
	>>> M.shape
	(5, 5)
	```
	"""
	names = list(clouds)
	m = len(names)
	W2_sq = np.zeros((m, m))

	# Precompute uniform weights once
	weights = [np.ones((clouds[n].shape[0],)) / clouds[n].shape[0] for n in names]

	# Iterate over unique index pairs
	for i, j in tqdm(list(combinations(range(m), 2)), desc="W2_sq computations"):
		C = ot.dist(clouds[names[i]], clouds[names[j]])  # cost matrix
		if sinkhorn_reg > 0:
			dist2 = ot.sinkhorn2(weights[i], weights[j], C, sinkhorn_reg)
		else:
			dist2 = ot.emd2(weights[i], weights[j], C)
		W2_sq[i, j] = W2_sq[j, i] = dist2

	return W2_sq  # type: ignore[return-value]


def compute_kde_distance_matrix(
	clouds: dict[str, np.ndarray],
	bandwidth: float = 1.0,
	grid_bins: int = 50,
) -> tuple[list[str], Float[np.ndarray, "n_models n_models"]]:
	"""Approximate pairwise distances between point-cloud densities via KDE + L2.

	# Parameters:
	 - `clouds: dict[str, np.ndarray]`
	    Mapping model → point-cloud array of shape (n_i, dim).
	 - `bandwidth: float`
	    KDE bandwidth for all kernels.
	 - `grid_bins: int`
	    Number of bins per dimension for the evaluation grid.

	# Returns:
	 - `Float[np.ndarray, \"n_models n_models\"]`
	    Symmetric matrix of L₂ differences between KDE densities.
	"""
	names = list(clouds)
	m = len(names)
	dims = next(iter(clouds.values())).shape[1]

	# 1. Fit KDEs
	kdes = []
	for model in names:
		kde = KernelDensity(bandwidth=bandwidth)
		kde.fit(clouds[model])
		kdes.append(kde)

	# 2. Build a common grid
	mins = np.vstack([clouds[n].min(axis=0) for n in names]).min(axis=0)
	maxs = np.vstack([clouds[n].max(axis=0) for n in names]).max(axis=0)
	axes = [np.linspace(mins[d], maxs[d], grid_bins) for d in range(dims)]
	mesh = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1)
	grid_points = mesh.reshape(-1, dims)  # (G, D), where G = grid_bins**dims

	# 3. Evaluate densities
	dens = np.zeros((m, grid_points.shape[0]))
	for i, kde in enumerate(kdes):
		log_d = kde.score_samples(grid_points)
		dens[i] = np.exp(log_d)

	# 4. Compute L2 distances
	Dmat = np.zeros((m, m))
	for i in range(m):
		for j in range(i + 1, m):
			diff = dens[i] - dens[j]
			d2 = np.sqrt(
				np.sum(diff**2)
				* np.prod([(maxs[d] - mins[d]) / (grid_bins - 1) for d in range(dims)])
			)
			Dmat[i, j] = Dmat[j, i] = d2

	return names, Dmat  # type: ignore[return-value]


def compute_kde_distance_matrix_torch(
	clouds: dict[str, np.ndarray],
	*,
	bandwidth: float = 0.5,
	grid_bins: int = 50,
	device: str | torch.device | None = None,
) -> dict[str, Float[torch.Tensor, "n_models n_models"]]:
	"""Pairwise KDE–L₂ distances on GPU/CPU with PyTorch.

	Uses an explicit Gaussian kernel:
	    p̂(x) = (1 / (n (2π)^{d/2} h^d)) ∑ₖ exp(‖x − x_k‖² / (−2 h²))

	The normalising constant cancels in the L₂ difference, so we drop it.

	# Parameters
	----------
	clouds : dict[str, np.ndarray]
	    Mapping model → point cloud (n_i, d).
	bandwidth : float, default=0.5
	    Gaussian kernel bandwidth *h*.
	grid_bins : int, default=50
	    Number of grid points per PCA dimension.
	device : str | torch.device | None
	    'cuda', 'cpu', etc. `None` → CUDA if available else CPU.

	# Returns
	-------
	dict with:
	    - "models": list[str]
	    - "distance_matrix": torch.Tensor (M, M)  (double on chosen device)
	"""
	if device is None:
		device = "cuda" if torch.cuda.is_available() else "cpu"

	names = list(clouds)
	m = len(names)
	d = next(iter(clouds.values())).shape[1]

	# ---- 1. Shared evaluation grid ------------------------------------------------
	print("\tBuilding grid...")
	mins = np.vstack([x.min(axis=0) for x in clouds.values()]).min(axis=0)
	maxs = np.vstack([x.max(axis=0) for x in clouds.values()]).max(axis=0)
	axes = [np.linspace(mins[k], maxs[k], grid_bins) for k in range(d)]
	mesh = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1)  # (*grid_bins, d)
	grid_pts: Float[torch.Tensor, "*g d"] = torch.from_numpy(mesh.reshape(-1, d)).to(
		device=device, dtype=torch.float64
	)
	g = grid_pts.shape[0]

	cell_vol = np.prod((maxs - mins) / (grid_bins - 1))  # ΔV for Riemann sum

	# ---- 2. KDE evaluation for each cloud -----------------------------------------
	dens = torch.empty((m, g), device=device, dtype=torch.float64)
	for idx, name in tqdm(enumerate(names), desc="KDE eval"):
		x: Float[torch.Tensor, "n d"] = torch.as_tensor(
			clouds[name], device=device, dtype=torch.float64
		)
		# squared pair-wise distances: (g, n)
		dist2: Float[torch.Tensor, "g n"] = torch.cdist(grid_pts, x) ** 2
		dens[idx] = torch.exp(-dist2 / (2.0 * bandwidth**2)).sum(dim=1) / x.shape[0]

	# ---- 3. Pairwise L₂ distances in one tensor op --------------------------------
	print("\tComputing L2 distances...")
	diff: Float[torch.Tensor, "m m g"] = dens.unsqueeze(0) - dens.unsqueeze(1)
	D_sq: Float[torch.Tensor, "m m"] = (diff**2).sum(dim=2) * cell_vol
	D: Float[torch.Tensor, "m m"] = torch.sqrt(D_sq)

	return {"models": names, "distance_matrix": D}

In [4]:
def plot_heatmap(
	matrix: Float[np.ndarray, "n n"],
	labels: list[str],
	title: str = "",
) -> None:
	"""Plot a square heatmap of a distance matrix.

	# Parameters:
	 - `matrix: Float[np.ndarray, "n n"]`
	    Square matrix of distances.
	 - `labels: List[str]`
	    Tick labels for rows/columns.
	 - `title: str`
	    Title of the plot.

	# Usage:

	```python
	>>> plot_heatmap(M, list(clouds.keys()))
	```
	"""
	fig, ax = plt.subplots()
	im = ax.imshow(matrix)
	ax.set_xticks(np.arange(len(labels)))
	ax.set_yticks(np.arange(len(labels)))
	ax.set_xticklabels(labels, rotation=90)
	ax.set_yticklabels(labels)
	ax.set_title(title)
	fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
	plt.tight_layout()
	plt.show()

In [5]:
# Example usage
clouds: dict[str, np.ndarray] = load_pointclouds(
	"../data/features/pca.jsonl",
	n_dims=4,
)
print(f"Loaded {len(clouds)} point clouds:")
for k, v in clouds.items():
	print(f"\t{k}: {v.shape}")
labels, dist_mat = compute_kde_distance_matrix_torch(clouds, grid_bins=16)
dbg_tensor(dist_mat)
plot_heatmap(dist_mat, list(clouds.keys()), title="KDE L2 distances")

Loaded 6 point clouds:
	gemma-2-2b: (26624, 4)
	gemma-2b: (18432, 4)
	Llama-3.2-1B: (65536, 4)
	gpt2-medium: (49152, 4)
	gpt2-small: (18432, 4)
	pythia-1b: (16384, 4)
	Building grid...


KDE eval: 0it [00:04, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 13.00 GiB. GPU 0 has a total capacity of 15.62 GiB of which 2.39 GiB is free. Including non-PyTorch memory, this process has 13.21 GiB memory in use. Of the allocated memory 13.01 GiB is allocated by PyTorch, and 8.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)